In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver
import os

In [2]:
load_dotenv

<function dotenv.main.load_dotenv(dotenv_path: str | ForwardRef('os.PathLike[str]') | None = None, stream: IO[str] | None = None, verbose: bool = False, override: bool = False, interpolate: bool = True, encoding: str | None = 'utf-8') -> bool>

In [3]:
llm = ChatGoogleGenerativeAI(
     model="gemini-3.6-flash",
     api_key=os.getenv("GOOGLE_API_KEY")
)

In [4]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [5]:
def generate_joke(state: JokeState):
    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt)

    return {'joke': response}


In [6]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [7]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)


checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [11]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic': 'pizza'}, config=config1)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


{'topic': 'pizza',
 'joke': AIMessage(content=[{'type': 'text', 'text': 'Why did the pizza go to the party by itself?\n\nBecause it was **provolone**!', 'extras': {'signature': 'ErAkCq0kARFNMg8up+PNC1tYPdDvH/xvNx17xRKnR4h7YSRJ15/BcUoyNBhd5aSaJEY4t2QihoK89zj7R/32VsaDHKOJ0/gmx84G+WFtSLwU8nAoy5ypTsq8zj5aUcqzCfQaT2b0cwtVzIrsxpbS07DqLCBjLMrdOruvsPQLItalazveN6W94Fo+ChsBss5zQJCvng5xibmvKx73XvEaQn+2C0rd6H0C51+VUEkJ7AnHN7oisBz2PfMFpbbVjK8XvrJV7WAEXMPw7hVXDsWh2BHBRskkecSN+L7M89zCum5z76eDE6cAITMPKQ4MktcYgYWvvXBvSTxMTwS0saQuDecMfPFdb+SYSulQdIcZ0X72YdQz40ZzTdfdJ5pJmBecz/Biza3UAGmQY8aiYbWFfD8AP6fz3vrKsaRrRh1PC9apJSum1tIAr3HnzxtHr/6WStGj2/1a9ce/aRjM90pmoROGi+RRvamXItr2a/lupHPhJ/oYgi9KUspNNlmFU4CrVLW04J7cBhr7C65Xh2lgExNhmHZUrp8yN7FVKOKhkNlk8URh8Muq21dAhX3igjLELgrv+/hhx8zdmYoQTJ8Hm7NCo/FaZxYwtWl4n/o1ARatHkw7VV7Zh5CqlpzEXeB+RA8jIbykKGeB4pLnMxlLenZX67AxIucHSvzuPO6GpnKfjrIVj1ffhT+XfDfSPbdBxelNxXytHceEYyKFwdK8Nz8DwrEI+/gwe474HeqKOY6pNqBDMyV7NbEhxTDiR/yOBSjajasGWGA57uDpXa/l2GGRji38V106qkqTI7cQ46OjvC2iGrDXWY

In [ ]:
workflow.get_state(config1)

In [ ]:
config2 = {"congfigurble": {"thread_id": "1"}}
workflow.invoke({'topic': 'pasta'}, config=config2)